1.Libraries


In [1]:
# @title
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import requests
from io import BytesIO
from datetime import datetime, timedelta
from openpyxl import load_workbook
from openpyxl.styles import Alignment, PatternFill

from concurrent.futures import ThreadPoolExecutor #for mulltiple reading of files

output_path = '/content/drive/MyDrive/Python projects/system_load.xlsx'

Mounted at /content/drive


2.Function to process ONE date

In [2]:
# @title
# Function to process ONE date
def process_date(current_date):

    date_str = current_date.strftime("%Y%m%d")
    date_print = current_date.strftime("%Y/%m/%d")

    year = current_date.strftime("%Y")
    month = current_date.strftime("%m")

    url = f"https://www.admie.gr/sites/default/files/attached-files/type-file/{year}/{month}/{date_str}_SystemRealizationSCADA_01.xls"

    try:
        response = requests.get(url, timeout=10)

        if response.status_code != 200:
            print(f"The date {date_print} doesn't exist.")
            return None

        file = BytesIO(response.content)

        df = pd.read_excel(file, sheet_name='System_Production', header=None, engine='xlrd')

        location = df[df.apply(
            lambda row: row.astype(str).str.contains("NET LOAD", case=False, na=False).any(),
            axis=1
        )]

        if location.empty:
            print(f"'NET LOAD' not found for {date_print}")
            return None

        row_index = location.index[0]

        values = df.iloc[row_index, 2:26].tolist()

        daily_sum = sum(values)

        row = [current_date.strftime("%Y-%m-%d")] + values + [daily_sum]

        print(f"{date_print} loaded")

        return row

    except:
        print(f"The date {date_print} failed")
        return None

3.Parallel Downloader

In [3]:
# @title
# Function for many downloads at the same time.
def get_net_load_parallel(start_date, end_date):

    date_list = []
    current_date = start_date

    while current_date <= end_date:
        date_list.append(current_date)
        current_date += timedelta(days=1)

    data_rows = []

    with ThreadPoolExecutor(max_workers=10) as executor:
        results = executor.map(process_date, date_list)

    for r in results:
        if r is not None:
            data_rows.append(r)

    return data_rows

4.Function for Excel Formatting

In [4]:
# @title
def format_excel(file_path):

    wb = load_workbook(file_path)
    ws = wb.active

    header_fill = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")
    center_align = Alignment(horizontal="center", vertical="center")

    # Format header
    for cell in ws[1]:
        cell.fill = header_fill
        cell.alignment = center_align

    # Center all values
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = center_align

    # Resize date column
    ws.column_dimensions['A'].width = 18

    wb.save(file_path)

5.Main Script

In [5]:
# @title
start_date = datetime(2025,1,1)
end_date = datetime(2026,4,9)

data_rows = get_net_load_parallel(start_date, end_date)

hours = [f"{str(h).zfill(2)}:30" for h in range(24)]
columns = ["Date"] + hours + ["Sum"]

df = pd.DataFrame(data_rows, columns=columns)

df.to_excel(output_path, index=False)

format_excel(output_path)

print("Excel file created successfully.")

2025/01/07 loaded2025/01/05 loaded
2025/01/01 loaded
2025/01/09 loaded
2025/01/03 loaded
2025/01/08 loaded
2025/01/04 loaded
2025/01/06 loaded

2025/01/10 loaded
2025/01/02 loaded
2025/01/12 loaded
2025/01/16 loaded
2025/01/15 loaded
2025/01/14 loaded
2025/01/11 loaded
2025/01/17 loaded
2025/01/13 loaded
2025/01/20 loaded
2025/01/19 loaded
2025/01/18 loaded
2025/01/21 loaded
2025/01/22 loaded
2025/01/26 loaded
2025/01/23 loaded
2025/01/27 loaded
2025/01/25 loaded
2025/01/28 loaded
2025/01/29 loaded
2025/01/30 loaded
2025/01/24 loaded
2025/02/02 loaded
2025/02/01 loaded
The date 2025/01/31 doesn't exist.
2025/02/03 loaded
2025/02/04 loaded
2025/02/07 loaded
2025/02/05 loaded
2025/02/08 loaded
2025/02/06 loaded
2025/02/10 loaded
2025/02/11 loaded
2025/02/12 loaded
2025/02/09 loaded
2025/02/13 loaded
2025/02/14 loaded
2025/02/16 loaded
2025/02/15 loaded
2025/02/18 loaded
2025/02/19 loaded
2025/02/17 loaded
2025/02/20 loaded
2025/02/22 loaded
2025/02/21 loaded
2025/02/24 loaded
2025/02/23 